# Preparing and Loading Text Data

A GPT model, at its heart, is a masterful number-crunching machine. It doesn't understand letters, words, or sentences; it only understands numerical patterns. Before we can train our model to write like Shakespeare, we must first become expert translators, converting the Bard's eloquent text into the language of numbers that our model can process.

By the end of this notebook, you will understand the entire journey of data from a raw text file to the perfectly shaped numerical batches that fuel the training process. You will be able to implement your own character-level tokenizer and write a `get_batch` function, the very component that feeds our hungry GPT model. This is the crucial step that connects the world of human language to the world of machine learning.

In [ ]:
import torch
import numpy as np
import time
from IPython.display import display, clear_output

# This setup cell contains all the imports we'll need for the notebook.
# We're keeping it minimal to focus on the concepts.

### The Core Idea: From a Library of Books to a Single Recipe

Imagine you want to teach a chef's apprentice to cook. You wouldn't just hand them a massive library of cookbooks and say, "Learn!" That's overwhelming and inefficient. Instead, you'd give them one simple recipe at a time.

Our data preparation process works the same way.

1.  **The Library (Raw Text):** We start with a giant text file, like the complete works of Shakespeare. This is our library of knowledge, but it's not in a usable format for our apprentice (the model).

2.  **Creating an Index (Tokenization):** First, we need a common language. We'll create a vocabulary of every unique character in the text ('A', 'B', 'C', 'a', 'b', 'c', '.', ' ', etc.). Then, we'll assign a unique number to each character. 'A' might become 0, 'B' becomes 1, and so on. This process of converting text into a sequence of numbers (tokens) is called **tokenization** or **numericalization**. We've essentially created an index for our library.

3.  **Writing Down the Recipes (Batching):** Now, instead of giving the model the entire multi-million-character text at once, we give it small, digestible "recipes." A single recipe looks like this:
    *   **Ingredients (Input `x`):** A short sequence of characters, like `"Thou art more"`
    *   **What to Cook Next (Target `y`):** The *exact same sequence*, but shifted one character to the right: `"hou art more "`

The model's job is to look at the ingredients (`x`) and learn to predict the very next ingredient (`y`). When it sees `"Thou ar"`, it must learn to predict `"t"`. When it sees `"Thou art"`, it must learn to predict `" "`. This simple, repeated task, when done millions of time, is what allows the model to learn the patterns, grammar, and style of the original text.

Our goal is to build the machine that generates these `(ingredients, next_ingredient)` recipes, which we call **batches**.

In [ ]:
# --- A Minimal, Self-Contained Data Loading Implementation ---

# Our raw text data
text = "The quick brown fox jumps over the lazy dog."

# 1. Create the vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Our vocabulary: {''.join(chars)}")
print(f"Vocabulary size: {vocab_size}")

# 2. Create the encoder and decoder
# This is our simple lookup table
stoi = { ch:i for i,ch in enumerate(chars) } # string to integer
itos = { i:ch for i,ch in enumerate(chars) } # integer to string
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# 3. Tokenize the entire dataset
data = torch.tensor(encode(text), dtype=torch.long)

# 4. The get_batch function
def get_batch(data, block_size, batch_size):
    """
    Selects random chunks of text to create input (x) and target (y) batches.
    """
    # Generate batch_size random starting points
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Create the input sequences (x)
    x = torch.stack([data[i:i+block_size] for i in ix])

    # Create the target sequences (y), which are shifted by one position
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    return x, y

### Walking Through the Code

Let's break down that implementation piece by piece. It's the blueprint for how nanoGPT handles data.

**Step 1: Create the Vocabulary**
```python
chars = sorted(list(set(text)))
vocab_size = len(chars)
```
We first find every unique character in our text. `set(text)` gives us the unique characters, and `sorted(list(...))` converts it into a sorted list. Sorting isn't strictly necessary, but it makes our vocabulary deterministic—it will be the same every time we run the code. The `vocab_size` is a critical number that we'll need later for our model's architecture.

**Step 2: Create the Encoder and Decoder**
```python
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
```
Here, we build our two-way translation dictionary.
*   `stoi` (string-to-integer) maps each character to its unique number (e.g., `' ' -> 0`, `'.' -> 1`, `'T' -> 2`).
*   `itos` (integer-to-string) does the reverse (e.g., `0 -> ' '`, `1 -> '.'`, `2 -> 'T'`).
The `encode` and `decode` functions are simple helpers that apply these lookups to entire strings or lists of numbers.

**Step 3: Tokenize the Dataset**
```python
data = torch.tensor(encode(text), dtype=torch.long)
```
We run our entire text through the `encode` function. This converts the full text, `"The quick brown fox..."`, into one long `torch.tensor` of integers, like `[2, 22, 10, 0, ... ]`. This tensor is our complete, pre-processed "library" of data, ready for training.

**Step 4: The `get_batch` function**
This is the most important part. It's the function that will be called at every single training step.
```python
def get_batch(data, block_size, batch_size):
    # ...
```
*   `ix = torch.randint(len(data) - block_size, (batch_size,))`: We need to pick `batch_size` random starting locations for our text snippets. We subtract `block_size` to ensure we don't pick a starting point so close to the end of the text that we can't form a full sequence.
*   `x = torch.stack([...])`: For each random starting index `i` in `ix`, we grab the slice of data from `i` up to `i+block_size`. This becomes one row in our input batch `x`. `torch.stack` combines these individual sequences into a single tensor.
*   `y = torch.stack([...])`: We do the exact same thing, but for the target `y`, we grab the slice from `i+1` to `i+block_size+1`. This creates the "shifted-by-one" relationship that is the foundation of GPT training.

In [ ]:
# --- Demonstration ---
# Let's see get_batch in action.

# Hyperparameters for our demonstration
# block_size is the length of the context for predictions
# batch_size is how many independent sequences we process in parallel
block_size = 8
batch_size = 4

# Fetch a single batch of data
x_batch, y_batch = get_batch(data, block_size=block_size, batch_size=batch_size)

print("--- Input Batch (x) ---")
print("Shape:", x_batch.shape)
print(x_batch)

print("\n--- Target Batch (y) ---")
print("Shape:", y_batch.shape)
print(y_batch)

print("\n--- Let's decode one example from the batch to see the relationship ---")
for i in range(batch_size):
    input_text = decode(x_batch[i].tolist())
    target_text = decode(y_batch[i].tolist())
    print(f"Example {i+1}:")
    print(f"  Input:  '{input_text}'")
    print(f"  Target: '{target_text}'")
    print("-" * 20)

### Going Deeper: Why Use Binary Files and `memmap`?

In our simple example, we held the entire tokenized dataset in a single tensor in RAM.
```python
data = torch.tensor(encode(text), dtype=torch.long)
```
This is fine for a few sentences, but what about the complete works of Shakespeare (1 million characters) or a massive dataset like OpenWebText (billions of characters)? You can't fit that into your computer's RAM!

The solution used in nanoGPT is both clever and efficient.

1.  **Save to a Binary File:** After tokenizing the text, the resulting long list of numbers is saved to a binary file. Instead of a `.txt` file, you get a `.bin` file. This is a very compact way to store a huge array of integers.

2.  **Use Memory-Mapping (`np.memmap`):** When it's time to train, instead of loading the entire file back into RAM, nanoGPT uses a technique called memory-mapping.

**Analogy:** Imagine you have a 10,000-page encyclopedia. Loading it into RAM is like trying to photocopy the entire thing and hold all the pages in your hands at once—impossible. Memory-mapping is like getting a library card for that encyclopedia. You don't hold any pages, but you have instant access. When you need to read page 5,432, the operating system (the librarian) fetches just that one page for you.

This is what `np.memmap` does. It creates a NumPy array that *points* to the data on your disk. When your `get_batch` function asks for a small chunk of data (e.g., from byte 1,000,000 to 1,000,008), the OS efficiently reads just that tiny piece from the disk into memory. This allows you to work with datasets that are far larger than your available RAM, which is essential for training large-scale models.

In [ ]:
# --- Visualization of the Batching Process ---

def visualize_batching(text, block_size, batch_size):
    # Re-use our encoding logic from before
    chars = sorted(list(set(text)))
    stoi = { ch:i for i,ch in enumerate(chars) }
    itos = { i:ch for i,ch in enumerate(chars) }
    encode = lambda s: [stoi[c] for c in s]
    data = encode(text)
    
    # Generate random starting indices
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # Step 1: Show the full text and its tokenized form
    print("--- The Full Data Sequence ---")
    print("Text:    ", text)
    print("Tokens:  ", data)
    print("\n" + "="*50 + "\n")
    
    time.sleep(2)
    
    for b in range(batch_size):
        # Clear previous output for an animation-like effect
        clear_output(wait=True)
        
        print(f"--- Creating Batch {b+1}/{batch_size} ---")
        
        start_index = ix[b]
        
        # Grab the x and y chunks
        x_tokens = data[start_index : start_index+block_size]
        y_tokens = data[start_index+1 : start_index+block_size+1]
        
        # --- Visualization ---
        # Build the visualization string for the full data
        viz_str = ""
        for i, token in enumerate(data):
            # Highlight x
            if start_index <= i < start_index + block_size:
                viz_str += f"\x1b[44;37m{token:2d}\x1b[0m " # Blue background for x
            # Highlight y (which overlaps with x)
            elif start_index+1 <= i < start_index + block_size + 1:
                viz_str += f"\x1b[42;37m{token:2d}\x1b[0m " # Green background for y
            else:
                viz_str += f"{token:2d} "
        
        print("Full Token Stream:")
        print(viz_str)
        print("Legend: \x1b[44;37m Input (x) \x1b[0m \x1b[42;37m Target (y) \x1b[0m")
        
        # Show the extracted batch
        print("\nExtracted Sequences:")
        print(f"Input  (x): {x_tokens}")
        print(f"Target (y): {y_tokens}")
        
        # Decode for clarity
        print("\nDecoded Sequences:")
        print(f"Input  (x): '{decode(x_tokens)}'")
        print(f"Target (y): '{decode(y_tokens)}'")
        
        print("\n" + "="*50 + "\n")
        time.sleep(2.5)

# Run the visualization
visualize_batching("The quick brown fox jumps over the lazy dog.", block_size=8, batch_size=4)

### Summary & What's Next

In this chapter, we transformed abstract text into a concrete format ready for model training. We took on the role of a data translator, ensuring our model gets a steady stream of well-formed "recipes" to learn from.

**What We Built:**
*   A character-level **tokenizer** to convert text into numerical IDs.
*   An **encoder** and **decoder** to translate back and forth between the human and machine world.
*   The `get_batch` function, which is the heart of the data pipeline, responsible for creating the `(input, target)` pairs that drive the learning process.

**Key Takeaways:**
*   **Models need numbers:** All text must be tokenized and numericalized before a model can process it.
*   **Prediction is structured as `(context -> next_token)`:** The learning task is defined by providing a sequence (`x`) and asking the model to predict the next token at every position (`y`).
*   **Efficiency matters for large datasets:** For datasets larger than RAM, saving tokenized data to a binary file and using memory-mapping (`np.memmap`) is a crucial optimization.

**What's Next:**
We have the two main components of our engine: the GPT architecture itself (from the previous notebook) and the fuel system to feed it data (this notebook). The next step is to put them together. In the next chapter, "Orchestrating the GPT Training Loop," we will write the code that starts the engine: it will repeatedly call `get_batch`, pass the data through the model, calculate how wrong the model's predictions are, and adjust the model's parameters to make it a little bit better, step by step.